In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append("../")

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import src.data_preprocessing.caption_generation as cg
import src.data_preprocessing.create_aux_data as cad
import src.data_preprocessing.data_utils as du
import src.data_preprocessing.gee_utils as gu
from src.data.base_caption_builder import BaseCaptionBuilder, DummyCaptionBuilder
from src.data.base_datamodule import BaseDataModule
from src.data.butterfly_caption_builder import ButterflyCaptionBuilder
from src.data.butterfly_dataset import ButterflyDataset

In [ ]:
datadir = "/Users/tplas/data/aether_data/s2bms/source/alphaearth_av-128/"
tiffiles = os.listdir(datadir)
tiffiles = [f for f in tiffiles if f.endswith(".tif")]
list_names = []
for i_f, f in tqdm(enumerate(tiffiles)):
    fp = os.path.join(datadir, f)
    im = du.load_tiff(fp)
    assert (
        im.ndim == 3 and im.shape[1] == 1 and im.shape[2] == 1
    ), f"Expected image to have shape (bands, 1, 1), but got {im.shape}"
    if i_f == 0:
        vals = np.squeeze(im)[:, None]
    else:
        vals = np.concatenate((vals, np.squeeze(im)[:, None]), axis=1)
    list_names.append("_".join(f.split("_")[1:4]))


n_embed = 64
assert (
    vals.shape[0] == n_embed
), f"Expected number of bands to be {n_embed}, but got {vals.shape[0]}"

In [ ]:
df_vals = pd.DataFrame(vals.T, columns=["emb_" + str(i) for i in range(n_embed)])
df_vals["name_loc"] = list_names
df_vals["ind_int"] = [int(x.split("_")[1]) for x in list_names]
df_vals = df_vals.sort_values("ind_int").reset_index(drop=True)
df_vals = df_vals.drop(columns=["ind_int"])
df_vals.to_csv(os.path.join(datadir, "aef-uk-unlabelled_average-128.csv"), index=False)

In [ ]:
df_vals